# Phase 3 — NLP & Transformers
## Day 12: Embeddings and Transformers
**Date:** 2026-04-24

### Learning Objectives
- Understand what word embeddings are and why they matter
- Learn the ideas behind Word2Vec (Skip-gram, CBOW) and GloVe
- Grasp the attention mechanism and why it replaced RNNs
- Understand the encoder-decoder architecture
- Get a high-level overview of BERT and how it differs from GPT

In [ ]:
# Setup
import numpy as np
from collections import Counter
import re
import math

## Sample Data

We'll create a small corpus to illustrate embedding concepts. No external files needed.

In [ ]:
# A small corpus of sentences about food and animals
corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the mouse",
    "the dog chased the cat",
    "the king wore a crown",
    "the queen wore a crown",
    "the king ruled the kingdom",
    "the queen ruled the kingdom",
    "a man ate an apple",
    "a woman ate an orange",
    "the cat ate the fish",
    "the dog ate the bone",
]

# Build vocabulary
all_words = " ".join(corpus).split()
vocab = sorted(set(all_words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size}")
print(f"Words: {vocab}")

---
## 1. Why Embeddings? The Problem with One-Hot Vectors

In Day 11, we used TF-IDF to represent documents as sparse vectors. But those vectors have a big problem: every word is equally distant from every other word. "cat" is just as far from "dog" as it is from "quantum". That's not how language works.

One-hot encoding gives each word its own dimension. With 50,000 words, each vector has 50,000 dimensions, and only one of them is 1. There's zero overlap between any two words. No notion of similarity at all.

In [ ]:
# One-hot encoding: every word is equally distant
def one_hot(word, word2idx, vocab_size):
    vec = np.zeros(vocab_size)
    vec[word2idx[word]] = 1.0
    return vec

cat_vec = one_hot("cat", word2idx, vocab_size)
dog_vec = one_hot("dog", word2idx, vocab_size)
crown_vec = one_hot("crown", word2idx, vocab_size)

def cosine_sim(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return np.dot(a, b) / denom

print(f"cat vs dog (one-hot):   {cosine_sim(cat_vec, dog_vec):.4f}")
print(f"cat vs crown (one-hot): {cosine_sim(cat_vec, crown_vec):.4f}")
print("\nBoth are 0.0. One-hot sees no similarity between any words.")

**Embeddings solve this.** Instead of a sparse 50,000-dimensional vector, we map each word to a dense vector with maybe 100-300 dimensions. Words that appear in similar contexts end up with similar vectors. "cat" and "dog" would be close. "cat" and "quantum" would be far.

---
## 2. Word2Vec: Learning Embeddings from Context

Word2Vec (Mikolov et al., 2013) was a breakthrough. The core idea is simple: **you shall know a word by the company it keeps.** If "cat" and "dog" appear near the same words (sat, chased, ate), they should have similar vectors.

There are two variants:
- **CBOW (Continuous Bag of Words):** Predict the center word from surrounding context words. Given ["the", "_", "sat"], predict "cat".
- **Skip-gram:** Predict the surrounding words from the center word. Given "cat", predict "the" and "sat".

Both are shallow neural networks with one hidden layer. The weights of that hidden layer become your word embeddings.

In [ ]:
# Let's build Skip-gram training pairs
# For each word, pair it with its neighbors within a window

def build_skipgram_pairs(corpus, word2idx, window=2):
    pairs = []
    for sentence in corpus:
        words = sentence.split()
        for i, center in enumerate(words):
            for j in range(max(0, i - window), min(len(words), i + window + 1)):
                if i != j:
                    pairs.append((center, words[j]))
    return pairs

pairs = build_skipgram_pairs(corpus, word2idx, window=2)
print(f"Total skip-gram pairs: {len(pairs)}")
print(f"\nFirst 10 pairs (center -> context):")
for center, ctx in pairs[:10]:
    print(f"  {center:10s} -> {ctx}")

In [ ]:
# A tiny Word2Vec from scratch (simplified Skip-gram with numpy)
# This is for learning. In practice, use gensim or other libraries.

embedding_dim = 10
np.random.seed(42)

# Two weight matrices: input embeddings (W1) and output embeddings (W2)
W1 = np.random.randn(vocab_size, embedding_dim) * 0.1  # input -> hidden
W2 = np.random.randn(embedding_dim, vocab_size) * 0.1  # hidden -> output

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

# Training loop (very simplified, no negative sampling)
learning_rate = 0.05
losses = []

for epoch in range(50):
    epoch_loss = 0
    for center_word, context_word in pairs:
        # Forward pass
        center_idx = word2idx[center_word]
        context_idx = word2idx[context_word]
        
        hidden = W1[center_idx]               # (embedding_dim,)
        output = softmax(hidden @ W2)          # (vocab_size,)
        
        # Loss (cross entropy)
        loss = -np.log(output[context_idx] + 1e-10)
        epoch_loss += loss
        
        # Backward pass
        output[context_idx] -= 1              # gradient of softmax + CE
        
        # Update W2
        W2 -= learning_rate * np.outer(hidden, output)
        # Update W1
        W1[center_idx] -= learning_rate * (output @ W2.T)
    
    losses.append(epoch_loss / len(pairs))

print(f"Loss after epoch 1:  {losses[0]:.4f}")
print(f"Loss after epoch 50: {losses[-1]:.4f}")
print("\nEmbeddings learned! W1 rows are our word vectors.")

In [ ]:
# Now let's check: are similar words close together?
def most_similar(word, W, word2idx, idx2word, top_n=5):
    if word not in word2idx:
        return []
    vec = W[word2idx[word]]
    sims = []
    for other_word, idx in word2idx.items():
        if other_word == word:
            continue
        sim = cosine_sim(vec, W[idx])
        sims.append((other_word, sim))
    sims.sort(key=lambda x: -x[1])
    return sims[:top_n]

print("Most similar to 'cat':")
for w, s in most_similar("cat", W1, word2idx, idx2word):
    print(f"  {w:12s} {s:.4f}")

print("\nMost similar to 'king':")
for w, s in most_similar("king", W1, word2idx, idx2word):
    print(f"  {w:12s} {s:.4f}")

print("\nNote: with such a tiny corpus, results won't be perfect.")
print("But you can see the model tries to group related words.")

### The Famous king - man + woman = queen

One of the coolest properties of Word2Vec: you can do arithmetic with word vectors. The relationship between "king" and "man" is similar to the relationship between "queen" and "woman". So `king - man + woman` should land near `queen`.

Our corpus is too small for this to work reliably, but let's try anyway.

In [ ]:
# Word arithmetic: king - man + woman = ?
result_vec = W1[word2idx["king"]] - W1[word2idx["man"]] + W1[word2idx["woman"]]

# Find closest word to this vector
sims = []
for word, idx in word2idx.items():
    if word in ["king", "man", "woman"]:
        continue
    sim = cosine_sim(result_vec, W1[idx])
    sims.append((word, sim))
sims.sort(key=lambda x: -x[1])

print("king - man + woman = ?")
for w, s in sims[:5]:
    print(f"  {w:12s} {s:.4f}")
print("\nWith a real Word2Vec model trained on billions of words,")
print("'queen' would be the top answer.")

---
## 3. GloVe: Global Vectors for Word Representation

GloVe (Pennington et al., 2014) takes a different approach. Instead of predicting context words one at a time (like Word2Vec), GloVe builds a **co-occurrence matrix** first. It counts how often each pair of words appears near each other across the entire corpus. Then it trains embeddings to reproduce those co-occurrence statistics.

The key insight: the **ratio** of co-occurrence probabilities carries meaning. If "ice" co-occurs with "solid" much more than "steam" does, that ratio tells you something about temperature and state of matter.

In practice, GloVe and Word2Vec give similar quality embeddings. GloVe is sometimes faster because it works on the global matrix rather than iterating over individual word pairs.

In [ ]:
# Build a co-occurrence matrix from our corpus
window = 2
cooccurrence = np.zeros((vocab_size, vocab_size))

for sentence in corpus:
    words = sentence.split()
    for i, w in enumerate(words):
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if i != j:
                cooccurrence[word2idx[w], word2idx[words[j]]] += 1

# Show co-occurrence for a few interesting words
interesting = ["cat", "dog", "king", "queen", "sat", "ate"]
print("Co-occurrence counts (how often words appear near each other):")
print(f"{'':12s}", end="")
for w in interesting:
    print(f"{w:>8s}", end="")
print()

for w1 in interesting:
    print(f"{w1:12s}", end="")
    for w2 in interesting:
        count = cooccurrence[word2idx[w1], word2idx[w2]]
        print(f"{count:8.0f}", end="")
    print()

print("\nNotice: cat and dog have similar patterns (both appear near 'sat', 'ate', etc.)")
print("Same for king and queen. That's the signal embeddings capture.")

---
## 4. The Attention Mechanism

Before Transformers, NLP used RNNs (Recurrent Neural Networks) and LSTMs to process sequences. These read text word by word, left to right. The problem: by the time you reach the end of a long sentence, the model has partially forgotten the beginning. Information gets compressed into a single hidden state, creating a bottleneck.

**Attention** (Bahdanau et al., 2014) solved this. Instead of forcing everything through one bottleneck, attention lets the model look back at ALL previous words and decide which ones are most relevant right now.

Think of it like highlighting important words in a textbook. When translating "The cat sat on the mat" to French, to translate "sat", the model should pay most attention to "cat" (the subject) and "sat" itself. It shouldn't waste attention on "the" or "on".

In [ ]:
# Simple attention mechanism from scratch
# Given a query and a set of key-value pairs, compute weighted sum of values

np.random.seed(42)

# Pretend we have 5 words, each represented by a 4-dimensional vector
seq_len = 5
d_model = 4

# Random "encoded" representations for: ["The", "cat", "sat", "on", "mat"]
words = ["The", "cat", "sat", "on", "mat"]
X = np.random.randn(seq_len, d_model)

# Step 1: Compute attention scores (dot product between each pair)
# How much should word i attend to word j?
scores = X @ X.T  # (seq_len, seq_len)

# Step 2: Scale by sqrt(d_model) to prevent huge values
scores_scaled = scores / np.sqrt(d_model)

# Step 3: Softmax to get attention weights (each row sums to 1)
def softmax_rows(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / e_x.sum(axis=1, keepdims=True)

attention_weights = softmax_rows(scores_scaled)

# Step 4: Weighted sum of values
output = attention_weights @ X  # (seq_len, d_model)

print("Attention weights (each row = how much that word attends to others):")
print(f"{'':6s}", end="")
for w in words:
    print(f"{w:>8s}", end="")
print()
for i, w in enumerate(words):
    print(f"{w:6s}", end="")
    for j in range(seq_len):
        print(f"{attention_weights[i, j]:8.3f}", end="")
    print()

print("\nEach word now has a context-aware representation that's a")
print("weighted mix of all other words in the sentence.")

### Query, Key, Value (QKV)

In Transformers, attention uses three separate projections of the input:
- **Query (Q):** What am I looking for?
- **Key (K):** What do I contain?
- **Value (V):** What information do I actually give back?

The formula: `Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V`

This is called **Scaled Dot-Product Attention**.

In [ ]:
# QKV Attention
np.random.seed(42)

d_k = 4  # dimension of queries and keys
d_v = 4  # dimension of values

# Learned projection matrices
W_Q = np.random.randn(d_model, d_k) * 0.5
W_K = np.random.randn(d_model, d_k) * 0.5
W_V = np.random.randn(d_model, d_v) * 0.5

# Project input into Q, K, V
Q = X @ W_Q  # (seq_len, d_k)
K = X @ W_K  # (seq_len, d_k)
V = X @ W_V  # (seq_len, d_v)

# Scaled dot-product attention
scores = Q @ K.T / np.sqrt(d_k)  # (seq_len, seq_len)
attn_weights = softmax_rows(scores)
attn_output = attn_weights @ V    # (seq_len, d_v)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print("Attention output shape:", attn_output.shape)
print("\nWith separate Q, K, V projections, the model learns")
print("WHAT to look for (Q), WHAT to advertise (K), and WHAT to give (V).")

### Multi-Head Attention

One attention head captures one type of relationship (maybe subject-verb). But sentences have many types of relationships at once. **Multi-head attention** runs several attention heads in parallel, each with its own Q/K/V projections, then concatenates the results.

For example, with 8 heads:
- Head 1 might learn syntactic relationships (subject-verb)
- Head 2 might learn positional relationships (nearby words)
- Head 3 might learn semantic relationships (synonyms)
- And so on

In [ ]:
# Multi-Head Attention (simplified)
np.random.seed(42)

n_heads = 2
d_k_per_head = d_model // n_heads  # split dimensions across heads

head_outputs = []
for h in range(n_heads):
    # Each head has its own projections
    Wq = np.random.randn(d_model, d_k_per_head) * 0.5
    Wk = np.random.randn(d_model, d_k_per_head) * 0.5
    Wv = np.random.randn(d_model, d_k_per_head) * 0.5
    
    q = X @ Wq
    k = X @ Wk
    v = X @ Wv
    
    s = q @ k.T / np.sqrt(d_k_per_head)
    w = softmax_rows(s)
    head_out = w @ v
    head_outputs.append(head_out)
    
    print(f"Head {h+1} attention weights:")
    for i, word in enumerate(words):
        top_attn = words[np.argmax(w[i])]
        print(f"  {word} attends most to: {top_attn} ({w[i][np.argmax(w[i])]:.3f})")
    print()

# Concatenate heads
multi_head_output = np.concatenate(head_outputs, axis=1)
print(f"Multi-head output shape: {multi_head_output.shape}")
print("Each head learned different attention patterns!")

---
## 5. The Encoder-Decoder Architecture

The original Transformer ("Attention Is All You Need", Vaswani et al., 2017) was designed for machine translation. It has two parts:

**Encoder:** Reads the input sentence and builds a rich representation. Uses self-attention so each word can see all other words. Stacks 6 identical layers, each with multi-head attention + feed-forward network.

**Decoder:** Generates the output sentence word by word. Also uses self-attention, but with a **mask** so each word can only see previous words (no peeking at the future). Additionally, it uses **cross-attention** to look at the encoder's output.

Both sides use residual connections and layer normalization to help training.

In [ ]:
# Causal mask: the decoder can't look at future tokens
# This is what makes autoregressive generation possible

seq_len = 5
causal_mask = np.tril(np.ones((seq_len, seq_len)))  # lower triangular

print("Causal mask (1 = can attend, 0 = blocked):")
print(f"{'':6s}", end="")
for w in words:
    print(f"{w:>8s}", end="")
print()
for i, w in enumerate(words):
    print(f"{w:6s}", end="")
    for j in range(seq_len):
        print(f"{causal_mask[i,j]:8.0f}", end="")
    print()

print("\n'The' can only see itself.")
print("'cat' can see 'The' and itself.")
print("'mat' can see all previous words.")

In [ ]:
# Masked attention: apply the causal mask before softmax
np.random.seed(42)

scores = X @ X.T / np.sqrt(d_model)

# Replace masked positions with -infinity so softmax gives them 0 weight
masked_scores = np.where(causal_mask == 1, scores, -1e9)

masked_weights = softmax_rows(masked_scores)

print("Masked attention weights (decoder style):")
print(f"{'':6s}", end="")
for w in words:
    print(f"{w:>8s}", end="")
print()
for i, w in enumerate(words):
    print(f"{w:6s}", end="")
    for j in range(seq_len):
        print(f"{masked_weights[i,j]:8.3f}", end="")
    print()

print("\nFuture positions get zero attention. This is how GPT generates text")
print("one token at a time without seeing the answer ahead of time.")

### Positional Encoding

Attention treats all positions equally. Unlike RNNs, it has no built-in sense of word order. "Dog bites man" and "Man bites dog" would look the same. To fix this, Transformers add **positional encodings** to the input embeddings.

The original paper used sine and cosine functions at different frequencies. Modern models (like BERT and GPT) often use learned positional embeddings instead.

In [ ]:
# Sinusoidal positional encoding
def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            PE[pos, i] = np.sin(pos / (10000 ** (i / d_model)))
            if i + 1 < d_model:
                PE[pos, i + 1] = np.cos(pos / (10000 ** (i / d_model)))
    return PE

PE = positional_encoding(seq_len=10, d_model=8)

print("Positional encoding for 10 positions, 8 dimensions:")
print("(Each row = one position, each column = one dimension)")
print()
for pos in range(10):
    vals = " ".join(f"{v:6.3f}" for v in PE[pos])
    print(f"  pos {pos}: [{vals}]")

print("\nLow-frequency dimensions change slowly across positions.")
print("High-frequency dimensions change rapidly.")
print("This gives each position a unique fingerprint.")

---
## 6. BERT vs GPT: Two Sides of the Transformer

The original Transformer had both encoder and decoder. Researchers quickly realized you could use just one half and get great results for different tasks.

**BERT (Bidirectional Encoder Representations from Transformers):** Uses only the encoder. Sees the entire input at once (bidirectional). Trained by masking random words and predicting them. Great for understanding tasks like classification, NER, and question answering.

**GPT (Generative Pre-trained Transformer):** Uses only the decoder. Sees text left-to-right (autoregressive). Trained by predicting the next word. Great for generation tasks like writing, summarization, and chat.

| Feature | BERT | GPT |
|---------|------|-----|
| Architecture | Encoder only | Decoder only |
| Direction | Bidirectional | Left-to-right |
| Pre-training | Masked Language Model (MLM) | Next token prediction |
| Best for | Classification, NER, QA | Text generation, chat |
| Sees future? | Yes | No |

In [ ]:
# BERT's Masked Language Model (MLM) training
# Mask 15% of tokens, then predict them

sentence = "the cat sat on the mat".split()

def mask_sentence(words, mask_prob=0.15, seed=42):
    np.random.seed(seed)
    masked = words.copy()
    targets = {}
    for i in range(len(masked)):
        if np.random.random() < mask_prob or i == 1:  # force mask 'cat' for demo
            targets[i] = masked[i]
            masked[i] = "[MASK]"
    return masked, targets

masked_sent, targets = mask_sentence(sentence)

print("Original: ", " ".join(sentence))
print("Masked:   ", " ".join(masked_sent))
print(f"Target:    predict that position {list(targets.keys())} = {list(targets.values())}")
print()
print("BERT sees the WHOLE sentence (including words after the mask).")
print("This is why it's 'bidirectional'. It uses both left and right context.")
print()
print("GPT would only see: 'the [predict next]' and never peek ahead.")

In [ ]:
# Comparing attention patterns: BERT (full) vs GPT (causal)
print("=" * 50)
print("BERT attention: every word sees every other word")
print("=" * 50)
bert_mask = np.ones((5, 5))
for i, w in enumerate(words):
    visible = [words[j] for j in range(5) if bert_mask[i, j] == 1]
    print(f"  {w:6s} sees: {', '.join(visible)}")

print()
print("=" * 50)
print("GPT attention: each word only sees previous words")
print("=" * 50)
for i, w in enumerate(words):
    visible = [words[j] for j in range(5) if causal_mask[i, j] == 1]
    print(f"  {w:6s} sees: {', '.join(visible)}")

---
## 7. Putting It All Together: The Transformer Block

A single Transformer block has these steps:
1. Multi-head self-attention
2. Add & normalize (residual connection + layer norm)
3. Feed-forward network (two linear layers with ReLU)
4. Add & normalize again

Stack 6-12 of these blocks, and you get BERT or GPT. Modern models like GPT-4 stack many more.

In [ ]:
# A simplified Transformer block in numpy
np.random.seed(42)

def layer_norm(x, eps=1e-6):
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def feed_forward(x, d_ff=8):
    """Two linear layers with ReLU in between"""
    W1_ff = np.random.randn(x.shape[-1], d_ff) * 0.5
    W2_ff = np.random.randn(d_ff, x.shape[-1]) * 0.5
    hidden = np.maximum(0, x @ W1_ff)  # ReLU
    return hidden @ W2_ff

def self_attention(X):
    d = X.shape[-1]
    scores = X @ X.T / np.sqrt(d)
    weights = softmax_rows(scores)
    return weights @ X

# One Transformer block
def transformer_block(X):
    # Step 1: Self-attention
    attn_out = self_attention(X)
    # Step 2: Add & Norm
    X = layer_norm(X + attn_out)
    # Step 3: Feed-forward
    ff_out = feed_forward(X)
    # Step 4: Add & Norm
    X = layer_norm(X + ff_out)
    return X

# Run our 5 words through 2 transformer blocks
X_input = np.random.randn(5, 4)  # 5 words, 4 dims

print("Input shape:", X_input.shape)
X_out = transformer_block(X_input)
print("After block 1:", X_out.shape)
X_out = transformer_block(X_out)
print("After block 2:", X_out.shape)
print("\nShape stays the same! Each block refines the representations.")
print("Deeper blocks capture more abstract patterns.")

---
## Tricky Bits

Common mistakes and confusions when learning about embeddings and transformers.

In [ ]:
# Tricky Bit 1: Embeddings are NOT the same as one-hot encodings
# A common confusion is thinking embeddings are just a lookup table.
# They ARE a lookup table, but the values are LEARNED to capture meaning.

print("One-hot for 'cat':")
print(one_hot("cat", word2idx, vocab_size))
print(f"  -> {vocab_size} dimensions, only one is nonzero")
print()
print("Embedding for 'cat' (from our Word2Vec):")
print(W1[word2idx["cat"]])
print(f"  -> {embedding_dim} dimensions, all are meaningful")
print()
print("The embedding IS a lookup table (row of a matrix),")
print("but the values encode semantic relationships.")

In [ ]:
# Tricky Bit 2: Self-attention has NO notion of word order by default
# Without positional encoding, "dog bites man" = "man bites dog"

# Shuffled input should give different results with positional encoding
np.random.seed(42)

X_original = np.random.randn(3, 4)  # ["dog", "bites", "man"]
X_shuffled = X_original[[2, 1, 0]]   # ["man", "bites", "dog"]

# Without positional encoding: attention is permutation equivariant
attn_orig = self_attention(X_original)
attn_shuf = self_attention(X_shuffled)

# The outputs are just shuffled versions of each other!
print("Without positional encoding:")
print(f"  Original attention output[0] (for 'dog'):  {attn_orig[0].round(3)}")
print(f"  Shuffled attention output[2] (for 'dog'):  {attn_shuf[2].round(3)}")
print(f"  Same? {np.allclose(attn_orig[0], attn_shuf[2])}")
print()
print("The model can't tell the difference between 'dog bites man'")
print("and 'man bites dog'. Positional encoding is essential!")

In [ ]:
# Tricky Bit 3: Scaling in attention matters a lot
# Without dividing by sqrt(d_k), softmax saturates

np.random.seed(42)
d_big = 512  # typical model dimension
q = np.random.randn(1, d_big)
k = np.random.randn(5, d_big)

scores_unscaled = (q @ k.T).flatten()
scores_scaled = (q @ k.T / np.sqrt(d_big)).flatten()

print("Unscaled scores:", scores_unscaled.round(2))
print("Scaled scores:  ", scores_scaled.round(2))
print()
print("Softmax of unscaled:", softmax(scores_unscaled).round(4))
print("Softmax of scaled:  ", softmax(scores_scaled).round(4))
print()
print("Without scaling, one score dominates and gradients vanish.")
print("Scaling keeps the distribution smooth so learning works.")

---
## Trick Questions

Test your understanding of embeddings and transformers.

**Q1:** Word2Vec learns two sets of embeddings (W1 and W2). Which one do people normally use?

<details><summary>Answer</summary>
Usually W1 (the input embeddings). Some people average W1 and W2, which can give slightly better results. W2 is the output/context embedding matrix.
</details>

**Q2:** Can BERT generate text like ChatGPT?

<details><summary>Answer</summary>
Not naturally. BERT is trained to fill in blanks (MLM), not to generate text left-to-right. You could force it to generate by iteratively masking and predicting, but it would be slow and awkward. GPT-style models are designed for generation.
</details>

**Q3:** Why does multi-head attention split the dimensions instead of running full-size attention multiple times?

<details><summary>Answer</summary>
Efficiency. If you have d_model=512 and 8 heads, each head works with 64 dimensions. The total computation is the same as a single full-size attention, but you get 8 different "perspectives" for free.
</details>

**Q4:** If you remove positional encoding from a Transformer, what breaks?

<details><summary>Answer</summary>
The model loses all sense of word order. "The cat ate the fish" and "The fish ate the cat" would produce identical representations. Self-attention is permutation equivariant without positional information.
</details>

**Q5:** In the encoder-decoder Transformer, what is cross-attention?

<details><summary>Answer</summary>
Cross-attention is where the decoder attends to the encoder's output. The queries come from the decoder, but the keys and values come from the encoder. This is how the decoder "reads" the input when generating output (e.g., during translation).
</details>

---
## Exercises

Fill in the `___` blanks and run each cell. The `assert` statements will tell you if you got it right.

In [ ]:
# Exercise 1: Create a one-hot vector for "dog"
dog_onehot = np.zeros(vocab_size)
dog_onehot[___] = 1.0  # fill in the index for "dog"

assert dog_onehot[word2idx["dog"]] == 1.0
assert dog_onehot.sum() == 1.0
print("Exercise 1 passed!")

In [ ]:
# Exercise 2: Compute cosine similarity between "king" and "queen" embeddings
king_emb = W1[word2idx["king"]]
queen_emb = W1[word2idx["queen"]]

dot_product = np.dot(king_emb, queen_emb)
norm_king = np.linalg.norm(___)
norm_queen = np.linalg.norm(___)
similarity = dot_product / (norm_king * norm_queen)

assert -1 <= similarity <= 1, "Cosine similarity must be between -1 and 1"
print(f"Exercise 2 passed! king-queen similarity: {similarity:.4f}")

In [ ]:
# Exercise 3: Build skip-gram pairs for a single sentence
# Given "I love NLP", with window=1, what are the pairs?
test_sentence = ["I", "love", "NLP"]
window = 1

# Fill in the expected pairs as a list of tuples
expected_pairs = ___  # hint: (center, context) for each word with its neighbors

assert len(expected_pairs) == 4, f"Should have 4 pairs, got {len(expected_pairs)}"
assert ("I", "love") in expected_pairs
assert ("love", "I") in expected_pairs
assert ("love", "NLP") in expected_pairs
assert ("NLP", "love") in expected_pairs
print("Exercise 3 passed!")

In [ ]:
# Exercise 4: Apply a causal mask to attention scores
np.random.seed(99)
scores_ex4 = np.random.randn(4, 4)

# Create a 4x4 lower triangular mask (1s on and below diagonal, 0s above)
mask_ex4 = np.___((4, 4))  # which numpy function creates a lower triangular matrix of ones?

# Apply mask: set masked positions to -1e9
masked_scores_ex4 = np.where(mask_ex4 == 1, scores_ex4, ___)

assert mask_ex4[0, 1] == 0, "Position (0,1) should be masked"
assert mask_ex4[2, 1] == 1, "Position (2,1) should be visible"
assert masked_scores_ex4[0, 3] == -1e9, "Future positions should be -inf"
print("Exercise 4 passed!")

In [ ]:
# Exercise 5: Compute scaled dot-product attention scores
np.random.seed(123)
d_k_ex5 = 64
Q_ex5 = np.random.randn(3, d_k_ex5)
K_ex5 = np.random.randn(3, d_k_ex5)

# Compute Q @ K^T, then scale by sqrt(d_k)
raw_scores = Q_ex5 @ K_ex5.___  # transpose K
scaled_scores = raw_scores / np.sqrt(___)

assert raw_scores.shape == (3, 3)
assert np.abs(scaled_scores).max() < np.abs(raw_scores).max(), "Scaling should reduce magnitudes"
print("Exercise 5 passed!")

In [ ]:
# Exercise 6: Layer normalization
# Layer norm normalizes across features (last axis) for each sample
x_ex6 = np.array([[4.0, 0.0, -4.0, 8.0]])

mean_val = x_ex6.mean(axis=___)
std_val = x_ex6.std(axis=___)
normed = (x_ex6 - mean_val) / (std_val + 1e-6)

assert np.abs(normed.mean()) < 0.01, "Mean should be ~0 after layer norm"
assert np.abs(normed.std() - 1.0) < 0.01, "Std should be ~1 after layer norm"
print(f"Exercise 6 passed! Normalized: {normed.round(4)}")

In [ ]:
# Exercise 7: Which model uses which attention pattern?
# Fill in "full" for bidirectional attention, "causal" for left-to-right only

bert_attention = "___"   # BERT's attention type
gpt_attention = "___"    # GPT's attention type

assert bert_attention == "full", "BERT sees all tokens (bidirectional)"
assert gpt_attention == "causal", "GPT only sees past tokens (autoregressive)"
print("Exercise 7 passed!")

---
## Solutions

<details><summary>Exercise 1</summary>

```python
dog_onehot[word2idx["dog"]] = 1.0
```
</details>

<details><summary>Exercise 2</summary>

```python
norm_king = np.linalg.norm(king_emb)
norm_queen = np.linalg.norm(queen_emb)
```
</details>

<details><summary>Exercise 3</summary>

```python
expected_pairs = [("I", "love"), ("love", "I"), ("love", "NLP"), ("NLP", "love")]
```
</details>

<details><summary>Exercise 4</summary>

```python
mask_ex4 = np.tril(np.ones((4, 4)))
masked_scores_ex4 = np.where(mask_ex4 == 1, scores_ex4, -1e9)
```
</details>

<details><summary>Exercise 5</summary>

```python
raw_scores = Q_ex5 @ K_ex5.T
scaled_scores = raw_scores / np.sqrt(d_k_ex5)
```
</details>

<details><summary>Exercise 6</summary>

```python
mean_val = x_ex6.mean(axis=-1)
std_val = x_ex6.std(axis=-1)
```
</details>

<details><summary>Exercise 7</summary>

```python
bert_attention = "full"
gpt_attention = "causal"
```
</details>

---
## Cumulative Review Exercises

Mixed exercises from Days 1-11. Fill in the blanks and run the cells.

In [ ]:
# Review 1 (Day 1 - Pandas): Filter a DataFrame
import pandas as pd

df_r = pd.DataFrame({"name": ["Alice", "Bob", "Charlie"], "age": [25, 30, 35]})
older = df_r[df_r[___] > 28]  # filter for age > 28

assert len(older) == 2
print("Review 1 passed!")

In [ ]:
# Review 2 (Day 2 - Numpy): Broadcasting
a_r2 = np.array([[1, 2, 3], [4, 5, 6]])
b_r2 = np.array([10, 20, 30])

result_r2 = a_r2 ___ b_r2  # element-wise addition using broadcasting

assert result_r2[0, 0] == 11
assert result_r2[1, 2] == 36
print("Review 2 passed!")

In [ ]:
# Review 3 (Day 3 - Data Cleaning): Handle missing values
df_r3 = pd.DataFrame({"val": [1.0, np.nan, 3.0, np.nan, 5.0]})
filled = df_r3["val"].___()  # fill NaN with the mean of the column

# Hint: first compute mean, then use fillna
filled = df_r3["val"].fillna(df_r3["val"].___())

assert filled.isna().sum() == 0
assert filled[1] == 3.0  # mean of [1, 3, 5] = 3
print("Review 3 passed!")

In [ ]:
# Review 4 (Day 4 - Python Core): List comprehension
squares = [x**___ for x in range(1, 6)]  # squares of 1 to 5

assert squares == [1, 4, 9, 16, 25]
print("Review 4 passed!")

In [ ]:
# Review 5 (Day 6 - Train/Test): What does stratify do in train_test_split?
# Fill in the answer
answer_r5 = "___"  # "preserves class proportions" or "random split" ?

assert answer_r5 == "preserves class proportions"
print("Review 5 passed!")

In [ ]:
# Review 6 (Day 7 - LogReg): Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-___))

assert abs(sigmoid(0) - 0.5) < 0.01
assert sigmoid(100) > 0.99
assert sigmoid(-100) < 0.01
print("Review 6 passed!")

In [ ]:
# Review 7 (Day 8 - Ensembles): Bagging vs Boosting
# Which trains models in parallel? Which trains sequentially?
bagging_trains = "___"     # "parallel" or "sequential"
boosting_trains = "___"    # "parallel" or "sequential"

assert bagging_trains == "parallel"
assert boosting_trains == "sequential"
print("Review 7 passed!")

In [ ]:
# Review 8 (Day 9 - Metrics): Precision formula
# precision = TP / (TP + ?)
TP, FP, FN = 80, 20, 10
precision = TP / (TP + ___)

assert abs(precision - 0.8) < 0.01
print(f"Review 8 passed! Precision: {precision:.2f}")

In [ ]:
# Review 9 (Day 10 - SHAP): What does a positive SHAP value mean?
answer_r9 = "___"  # "pushes prediction higher" or "pushes prediction lower"?

assert answer_r9 == "pushes prediction higher"
print("Review 9 passed!")

In [ ]:
# Review 10 (Day 11 - TF-IDF): What does IDF measure?
# IDF = log(total_docs / docs_containing_term)
# A word appearing in ALL documents gets IDF = ?

total_docs = 100
docs_with_word = 100  # appears in every document

idf_value = np.log(total_docs / docs_with_word)
print(f"IDF when word appears in all docs: {idf_value:.4f}")

expected_idf = ___  # what is log(100/100)?
assert abs(idf_value - expected_idf) < 0.01
print("Review 10 passed! Common words get low IDF (close to 0).")

### Cumulative Review Solutions

<details><summary>Review 1</summary>

```python
older = df_r[df_r["age"] > 28]
```
</details>

<details><summary>Review 2</summary>

```python
result_r2 = a_r2 + b_r2
```
</details>

<details><summary>Review 3</summary>

```python
filled = df_r3["val"].fillna(df_r3["val"].mean())
```
</details>

<details><summary>Review 4</summary>

```python
squares = [x**2 for x in range(1, 6)]
```
</details>

<details><summary>Review 5</summary>

```python
answer_r5 = "preserves class proportions"
```
</details>

<details><summary>Review 6</summary>

```python
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
```
</details>

<details><summary>Review 7</summary>

```python
bagging_trains = "parallel"
boosting_trains = "sequential"
```
</details>

<details><summary>Review 8</summary>

```python
precision = TP / (TP + FP)
```
</details>

<details><summary>Review 9</summary>

```python
answer_r9 = "pushes prediction higher"
```
</details>

<details><summary>Review 10</summary>

```python
expected_idf = 0.0  # log(1) = 0
```
</details>

In [ ]:
# Cheat Sheet: Embeddings & Transformers
cheat_sheet = """
╔══════════════════════════════════════════════════════════════════╗
║           EMBEDDINGS & TRANSFORMERS CHEAT SHEET                ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  EMBEDDINGS                                                    ║
║  ──────────                                                    ║
║  One-hot:    sparse, no similarity info, huge dimensions       ║
║  Word2Vec:   dense, learned from context, captures similarity  ║
║    CBOW:     predict center word from context                  ║
║    Skip-gram: predict context from center word                 ║
║  GloVe:      uses global co-occurrence matrix                  ║
║  Arithmetic: king - man + woman ≈ queen                       ║
║                                                                ║
║  ATTENTION                                                     ║
║  ─────────                                                     ║
║  Formula:  Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) @ V   ║
║  Q = query (what am I looking for?)                            ║
║  K = key   (what do I contain?)                                ║
║  V = value (what info do I give back?)                         ║
║  Multi-head: run h attention heads in parallel, concatenate    ║
║                                                                ║
║  TRANSFORMER BLOCK                                             ║
║  ─────────────────                                             ║
║  1. Multi-head self-attention                                  ║
║  2. Add & LayerNorm                                            ║
║  3. Feed-forward (2 linear + ReLU)                             ║
║  4. Add & LayerNorm                                            ║
║                                                                ║
║  BERT vs GPT                                                   ║
║  ───────────                                                   ║
║  BERT: encoder-only, bidirectional, MLM, for understanding     ║
║  GPT:  decoder-only, causal mask, next-token, for generation   ║
║                                                                ║
║  POSITIONAL ENCODING                                           ║
║  ────────────────────                                          ║
║  Adds position info since attention has no built-in order      ║
║  Original: sin/cos at different frequencies                    ║
║  Modern: learned positional embeddings                         ║
║                                                                ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(cheat_sheet)

---
## Next up: Day 13 — HuggingFaceBasics

Tomorrow we'll use the Hugging Face library to load real pretrained models with just a few lines of code. You'll use `pipeline()`, `AutoTokenizer`, and `AutoModel` to do NLP tasks without training anything from scratch.